# 06 — End to end: sample, certify, synthesize, verify

This notebook runs the full pipeline for several families:

1. **sample** a certified MPU (`mpu_generators.build_family`);
2. **certify** exact unitarity for every $N$ (translation-invariant cutoff $\chi^2+1$);
3. **synthesize** the circuit with the cheapest boundary `TreeMPU` finds;
4. **verify** the circuit against the dense operator at small $N$;
5. **extrapolate** resources to large $N$ with the analytic recursion.

In [1]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import warnings

from mpu_generators import build_family
from TreeMPU import (
    UniformMPU,
    check_mpo_unitarity,
    dense_unitary_from_tensors,
    phase_aligned_error,
    reconstruct_circuit_unitary,
)

In [2]:
def pipeline(name, seed=0, N_verify=4):
    t, l, r = build_family(name, d=2, bond_dim=2, seed=seed)
    A = t[0]
    chi = A.shape[2]
    check_mpo_unitarity(chi**2 + 1, A, l, r)  # certificate for every N

    best = None
    for options in (dict(), dict(boundary_opt_k=1)):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = UniformMPU(A=A, l_vec=l, r_vec=r, N=N_verify, light=True, **options)
        if best is None or m.q_unif < best.q_unif - 1e-9:
            best = m

    qc, sys_regs, *_ = best.synthesize()
    U_rec, leak = reconstruct_circuit_unitary(qc, sys_regs)
    err = phase_aligned_error(U_rec, dense_unitary_from_tensors([A] * N_verify, l, r))
    params = best.synthesis_cost_params()
    return best, qc, err, leak, params


results = {}
for name in ["branching_uniform", "fusion_cyclic", "grafted"]:
    best, qc, err, leak, params = pipeline(name)
    results[name] = (best, params)
    print(
        f"{name:18s} boundary={best.boundary_source:12s} q={best.q_unif:.4f} ell={best.ell} | "
        f"N=4 circuit: {qc.num_qubits} qubits, error {err:.1e}, leakage {leak:.1e}"
    )

branching_uniform  boundary=qD_general   q=2.0000 ell=1 | N=4 circuit: 16 qubits, error 1.0e-14, leakage 2.4e-14


fusion_cyclic      boundary=qD_general   q=2.0000 ell=1 | N=4 circuit: 16 qubits, error 1.6e-14, leakage 3.2e-14


grafted            boundary=qD_general   q=2.0000 ell=1 | N=4 circuit: 16 qubits, error 5.6e-15, leakage 1.3e-14


In [3]:
print(f"{'family':18s} {'N':>7s} {'depth':>14s} {'gates':>18s} {'qubits':>8s}")
for name, (_best, params) in results.items():
    for N in (2**10, 2**16):
        st = UniformMPU.analytic_synthesis_stats(params, N)
        print(
            f"{name:18s} {N:7d} {st['depth']:14,d} {st['gate_count']:18,d} {st['num_qubits']:8,d}"
        )

family                   N          depth              gates   qubits
branching_uniform     1024      6,360,865      3,723,611,514    5,116
branching_uniform    65536  4,680,723,733 173,964,712,397,658  327,676
fusion_cyclic         1024      6,360,865      3,723,611,514    5,116


fusion_cyclic        65536  4,680,723,733 173,964,712,397,658  327,676
grafted               1024      6,360,865      3,723,611,514    5,116
grafted              65536  4,680,723,733 173,964,712,397,658  327,676


All three families reach $q=\chi=2$, hence $\ell=1$. At $d=\chi=2$ they then give **identical** resources: once $(d,\chi,\ell)$ are fixed, the tree structure, the register sizes and the (generic, dense) local isometries all have the same cost, whichever family the tensor came from. The depth grows as $N^{\log_2 3}$ and the gate count as $N^{\log_2 3+1}$ (each layer replays its children $2\ell+1$ times, on two children).

### Where to go from here
* `docs/userguide.md`: background, conventions, and what is proved vs. numerically validated.
* `docs/theory_code_map.md`: thesis/paper equation labels mapped to functions.
